# Procesamiento de Lenguaje Natural - Desafio 2
## Custom embeddings con Gensim - Radiohead

En este desafio se entrenan embeddings Word2Vec utilizando las letras de canciones de **Radiohead** como corpus.

In [1]:
import os
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from sklearn.manifold import TSNE

/Users/mferrari/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1 - Carga de datos

Cargamos las letras de canciones de Radiohead. Cada linea del archivo de texto representa una oracion/verso.

In [2]:
df_radiohead = pd.read_csv('../songs_dataset/radiohead.txt', sep='/n', header=None, engine='python')
print(f'Cantidad de documentos (versos): {df_radiohead.shape[0]}')
df_radiohead.head(10)

Cantidad de documentos (versos): 2343


,0
0,"Come on, come on"
1,You think you drive me crazy
2,"Come on, come on"
3,You and whose army?
4,You and your cronies
5,"Come on, come on"
6,Holy Roman empire
7,Come on if you think
8,Come on if you think
9,You can take us on


## 2 - Preprocesamiento

Convertimos cada verso en una lista de tokens (palabras en minuscula, sin puntuacion).

In [3]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

oraciones_tokenizadas = []
for _, fila in df_radiohead.iterrows():
    oraciones_tokenizadas.append(text_to_word_sequence(fila[0]))

print(f'Cantidad de oraciones: {len(oraciones_tokenizadas)}')
print(f'Ejemplo: {oraciones_tokenizadas[:3]}')

Cantidad de oraciones: 2343
Ejemplo: [['come', 'on', 'come', 'on'], ['you', 'think', 'you', 'drive', 'me', 'crazy'], ['come', 'on', 'come', 'on']]


## 3 - Entrenar Word2Vec

Entrenamos un modelo Word2Vec con arquitectura Skip-gram sobre el corpus de Radiohead. Skip-gram predice las palabras de contexto a partir de una palabra central, lo que suele generar mejores representaciones para palabras poco frecuentes.

In [4]:
class LossCallback(CallbackAny2Vec):
    """Callback para imprimir el loss en cada epoca."""
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print(f'Loss epoca {self.epoch}: {loss}')
        else:
            print(f'Loss epoca {self.epoch}: {loss - self.loss_anterior}')
        self.epoch += 1
        self.loss_anterior = loss

In [5]:
modelo_w2v = Word2Vec(
    min_count=5,
    window=3,
    vector_size=300,
    negative=20,
    workers=1,
    sg=1
)

modelo_w2v.build_vocab(oraciones_tokenizadas)
print(f'Cantidad de docs en el corpus: {modelo_w2v.corpus_count}')
print(f'Cantidad de palabras distintas en el vocabulario: {len(modelo_w2v.wv.index_to_key)}')

Cantidad de docs en el corpus: 2343
Cantidad de palabras distintas en el vocabulario: 383


In [6]:
modelo_w2v.train(
    oraciones_tokenizadas,
    total_examples=modelo_w2v.corpus_count,
    epochs=50,
    compute_loss=True,
    callbacks=[LossCallback()]
)

Loss epoca 0: 92139.1953125
Loss epoca 1: 45462.8203125
Loss epoca 2: 43467.328125
Loss epoca 3: 42692.390625
Loss epoca 4: 43710.234375
Loss epoca 5: 43293.15625
Loss epoca 6: 41131.5625
Loss epoca 7: 40990.375
Loss epoca 8: 40744.46875
Loss epoca 9: 39449.25
Loss epoca 10: 38480.15625
Loss epoca 11: 37451.0625
Loss epoca 12: 35613.25
Loss epoca 13: 35075.4375
Loss epoca 14: 34136.5625
Loss epoca 15: 33294.3125
Loss epoca 16: 32582.75
Loss epoca 17: 31773.375
Loss epoca 18: 31623.25
Loss epoca 19: 31256.0
Loss epoca 20: 30936.5625
Loss epoca 21: 29309.8125
Loss epoca 22: 28734.625
Loss epoca 23: 28965.9375
Loss epoca 24: 28249.0625
Loss epoca 25: 28164.625
Loss epoca 26: 27731.75
Loss epoca 27: 27964.125
Loss epoca 28: 25679.6875
Loss epoca 29: 24494.25
Loss epoca 30: 24382.5
Loss epoca 31: 24482.25
Loss epoca 32: 24877.125
Loss epoca 33: 24598.75
Loss epoca 34: 24045.875
Loss epoca 35: 24357.375
Loss epoca 36: 23556.25
Loss epoca 37: 24050.875
Loss epoca 38: 23515.0
Loss epoca 39: 22

(292253, 590150)

El loss muestra una tendencia descendente clara a lo largo de las 50 epocas (de 92139 a ~22500), lo que indica que el modelo esta aprendiendo las relaciones entre palabras del corpus. A partir de la epoca 30 aproximadamente, el loss comienza a fluctuar entre epocas (por ejemplo, epoca 48: 22297, epoca 49: 22568), lo cual es un comportamiento normal del entrenamiento estocastico con SGD y sugiere que el modelo esta cerca de converger.

## 4 - Explorar terminos similares y disimiles

Elegimos terminos relevantes del universo lirico de Radiohead para explorar las relaciones que capturo el modelo.

In [7]:
print("Palabras mas similares a 'everything':")
for palabra, similitud in modelo_w2v.wv.most_similar(positive=['everything'], topn=10):
    print(f'  {palabra:15s} | sim={similitud:.3f}')

Palabras mas similares a 'everything':
  messed          | sim=0.883
  time            | sim=0.844
  really          | sim=0.766
  happening       | sim=0.761
  killing         | sim=0.744
  first           | sim=0.742
  ah              | sim=0.717
  children        | sim=0.710
  eye             | sim=0.705
  walls           | sim=0.702


In [8]:
print("Palabras mas similares a 'alone':")
for palabra, similitud in modelo_w2v.wv.most_similar(positive=['alone'], topn=10):
    print(f'  {palabra:15s} | sim={similitud:.3f}')

Palabras mas similares a 'alone':
  left            | sim=0.870
  inside          | sim=0.795
  through         | sim=0.748
  bit             | sim=0.741
  space           | sim=0.726
  empty           | sim=0.707
  makes           | sim=0.692
  leave           | sim=0.690
  high            | sim=0.684
  open            | sim=0.674


In [9]:
print("Palabras mas similares a 'rain':")
for palabra, similitud in modelo_w2v.wv.most_similar(positive=['rain'], topn=10):
    print(f'  {palabra:15s} | sim={similitud:.3f}')

Palabras mas similares a 'rain':
  hearts          | sim=0.914
  broken          | sim=0.899
  make            | sim=0.857
  black           | sim=0.818
  fire            | sim=0.766
  blame           | sim=0.760
  door            | sim=0.759
  wears           | sim=0.716
  houses          | sim=0.712
  new             | sim=0.705


In [10]:
print("Palabras menos similares a 'love':")
for palabra, similitud in modelo_w2v.wv.most_similar(negative=['love'], topn=10):
    print(f'  {palabra:15s} | sim={similitud:.3f}')

Palabras menos similares a 'love':
  uptight         | sim=-0.087
  get             | sim=-0.109
  day             | sim=-0.116
  over            | sim=-0.128
  here            | sim=-0.141
  am              | sim=-0.142
  body            | sim=-0.146
  some            | sim=-0.149
  rest            | sim=-0.149
  off             | sim=-0.152


In [11]:
print("Palabras mas similares a 'light':")
for palabra, similitud in modelo_w2v.wv.most_similar(positive=['light'], topn=10):
    print(f'  {palabra:15s} | sim={similitud:.3f}')

Palabras mas similares a 'light':
  cat             | sim=0.663
  try             | sim=0.637
  gone            | sim=0.621
  fall            | sim=0.621
  clear           | sim=0.599
  words           | sim=0.598
  sea             | sim=0.594
  eye             | sim=0.585
  earth           | sim=0.584
  hand            | sim=0.579


### Interpretacion de los terminos similares

Los resultados reflejan el universo lirico de Radiohead de forma coherente:

- **'everything'** → 'messed' (sim=0.883), 'time' (0.844), 'really' (0.766), 'happening' (0.761), 'killing' (0.744) — las palabras mas similares estan asociadas a contextos de caos y desorden emocional, temas recurrentes en Radiohead. La alta similaridad con 'messed' indica que ambas aparecen en versos con carga emocional similar.

- **'alone'** → 'left' (sim=0.870), 'inside' (0.795), 'through' (0.748), 'empty' (0.707), 'leave' (0.690) — varias de estas palabras ('left', 'inside', 'empty', 'leave') estan asociadas al aislamiento y el vacio. Sin embargo, otras como 'bit' (0.741), 'through' (0.748) y 'high' (0.684) no tienen una relacion semantica tan clara con 'alone', lo que sugiere que la co-ocurrencia en versos cercanos domina sobre el significado.

- **'rain'** → 'hearts' (sim=0.914), 'broken' (0.899), 'make' (0.857), 'black' (0.818), 'fire' (0.766) — la asociacion mas fuerte de todo el analisis. Radiohead usa 'rain' como metafora emocional, no como referencia climatica, lo que explica su cercania con 'hearts', 'broken' y 'fire'.

- **'light'** → 'cat' (sim=0.663), 'try' (0.637), 'gone' (0.621), 'fall' (0.621), 'clear' (0.599), 'sea' (0.594), 'eye' (0.585), 'earth' (0.584) — las similaridades son notablemente mas bajas que en los otros terminos. Algunas palabras como 'sea', 'earth', 'eye', 'clear' forman un campo semantico de elementos naturales/sensoriales, pero otras como 'cat' y 'try' no tienen relacion semantica obvia, sugiriendo que 'light' se usa en contextos mas variados dentro del corpus.

- **Disimiles a 'love'** → 'uptight' (sim=-0.087), 'get' (-0.109), 'day' (-0.116), 'over' (-0.128), 'body' (-0.146) — las similaridades negativas son muy cercanas a cero, lo que indica que no hay palabras realmente opuestas a 'love' en el corpus, sino palabras que simplemente aparecen en contextos diferentes.

## 5 - Visualizacion de embeddings en 2D

Reducimos la dimensionalidad de los embeddings de 300 a 2 dimensiones usando t-SNE para poder visualizarlos.

In [12]:
def reducir_dimensiones(modelo, num_dimensiones=2):
    vectores = np.asarray(modelo.wv.vectors)
    etiquetas = np.asarray(modelo.wv.index_to_key)
    tsne = TSNE(n_components=num_dimensiones, random_state=42, perplexity=30)
    vectores_reducidos = tsne.fit_transform(vectores)
    return vectores_reducidos, etiquetas

In [13]:
import plotly.express as px

vectores_2d, etiquetas = reducir_dimensiones(modelo_w2v)

MAX_WORDS = 100

fig = px.scatter(
    x=vectores_2d[:MAX_WORDS, 0],
    y=vectores_2d[:MAX_WORDS, 1],
    text=etiquetas[:MAX_WORDS],
    title='Embeddings de Radiohead proyectados en 2D con t-SNE'
)
fig.update_traces(textposition='top center', marker=dict(size=5))
fig.update_layout(
    width=1400,
    height=1000,
    xaxis=dict(dtick=5),
    yaxis=dict(dtick=5)
)
fig.show()

### Interpretacion de los grupos observados

En la visualizacion 2D se pueden identificar agrupaciones coherentes con los resultados de similaridad obtenidos en el punto anterior:

- 'rain' y 'broken' aparecen como un par aislado en la parte superior del grafico, lo cual es consistente con su alta similaridad (sim=0.899).
- 'don't', 'leave', 'arms', 'hurt' forman un grupo compacto asociado a vulnerabilidad y dolor emocional.
- 'everything', 'really', 'time' aparecen juntas a la derecha, reflejando la similaridad observada entre 'everything' y 'time' (sim=0.844).
- 'more', 'lies', "there'll" se agrupan en la zona superior izquierda.
- 'out', 'run', 'again', 'take' forman un grupo de verbos de movimiento a la izquierda.
- 'your', 'in', 'into' aparecen juntas abajo, asociadas a espacios internos.

En general, los clusters del grafico refuerzan lo observado en las consultas de similaridad: las palabras que el modelo reporto como similares efectivamente aparecen cercanas en la proyeccion 2D. La zona central es mas difusa, con palabras funcionales de alta frecuencia dispersas, lo cual es esperable al aparecer en contextos muy variados.

In [14]:
vectores_3d, etiquetas_3d = reducir_dimensiones(modelo_w2v, 3)

fig_3d = px.scatter_3d(
    x=vectores_3d[:MAX_WORDS, 0],
    y=vectores_3d[:MAX_WORDS, 1],
    z=vectores_3d[:MAX_WORDS, 2],
    text=etiquetas_3d[:MAX_WORDS],
    title='Embeddings de Radiohead proyectados en 3D con t-SNE'
)
fig_3d.update_traces(marker_size=3)
fig_3d.update_layout(width=1000, height=700)
fig_3d.show()

## Conclusiones

- Se entreno un modelo Word2Vec Skip-gram sobre 2343 versos de Radiohead, obteniendo un vocabulario de 383 palabras con embeddings de 300 dimensiones. El loss mostro una tendencia descendente clara (de 92139 a ~22500) con fluctuaciones a partir de la epoca 30, propias del entrenamiento estocastico.
- Las consultas de similaridad mostraron resultados variados. La asociacion mas fuerte fue 'rain' → 'hearts' (sim=0.914), donde el modelo capturo el uso metaforico que Radiohead hace de elementos naturales. Para 'alone', palabras como 'left', 'inside', 'empty' son coherentes con aislamiento, pero otras como 'bit' y 'through' reflejan mas la co-ocurrencia en versos que una relacion semantica. Para 'light', las similaridades fueron las mas bajas (0.58-0.66) y aparecen resultados poco intuitivos como 'cat', lo que indica que el termino se usa en contextos muy variados.
- La visualizacion 2D con t-SNE mostro clusters verificables en el grafico: un grupo compacto de interpelacion directa (children, alive, attention, yourself), otro de vulnerabilidad (don't, leave, hurt, myself), uno de interioridad (head, my, your, inside, into), y uno de negatividad (nothing, wrong, over). En la zona central, palabras funcionales como 'the', 'of', 'like' se agrupan con palabras de relaciones emocionales ('home', 'love', 'feel', 'keep'), sugiriendo que adquieren un sesgo contextual dentro del corpus.
- El corpus reducido (2343 versos, 383 palabras) limita la calidad de los embeddings: con pocas ocurrencias por palabra, algunas asociaciones capturan ruido estadistico mas que relaciones semanticas genuinas. Esto se evidencia en las similaridades menos intuitivas y en las fluctuaciones del loss al final del entrenamiento.